# AIラジオ大学｜ワンクリック音声化（Irodori-TTS・無料）

Googleドライブの **制作ボード** で「音声化待ち」の講義を、無料の音声合成 **Irodori-TTS** でMP3にし、ドライブの **「AIラジオ大学 制作室 / 音声」** に保存します。ボードは「登録待ち」に進み、会議ログにも記録されます。

声づくりは、muse-irodori-voice（カガミカミさん）の知見に沿っています：**キャプションで声を設計 → 候補の中から選ぶ → 選んだ声を参照に固定**。一度決めれば、どの講義も同じ教授の声で話します。

**使い方**
1. メニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選んで保存（無料）
2. ①から順に ▶ を押す。②③は**最初の1回だけ**（教授の声を決める）。2回目からは ① → ④ → ⑤ → ⑥ でOK
3. Googleの許可画面が出たら「許可」

※ 特定の声優やキャラクターの声に寄せる使い方はしないでください。声の好みは「低め」「落ち着いた」のような特徴の言葉で書きます。


In [ ]:
#@title ① 準備とGoogle連携（毎回・初回は5〜10分）
import torch, google.auth, gspread
from google.colab import auth
from googleapiclient.discovery import build
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし（ランタイムをT4 GPUにしてください）')
!pip -q install uv
![ -d /content/Irodori-TTS ] || git clone -q https://github.com/Aratako/Irodori-TTS.git /content/Irodori-TTS
!cd /content/Irodori-TTS && uv sync -q --extra cu128
!wget -q -O /content/Irodori-TTS/irodori_worker.py https://raw.githubusercontent.com/bananaraw123/ai-radio-uni/main/tools/irodori_worker.py
auth.authenticate_user()
creds, _ = google.auth.default()
drive, docs, gc = build('drive', 'v3', credentials=creds), build('docs', 'v1', credentials=creds), gspread.authorize(creds)
def find(name, mime, parent=None, create=False):
    q = f"name = '{name}' and mimeType = '{mime}' and trashed = false" + (f" and '{parent}' in parents" if parent else '')
    r = drive.files().list(q=q, fields='files(id)').execute()['files']
    if r: return r[0]['id']
    if create: return drive.files().create(body={'name': name, 'parents': [parent], 'mimeType': mime}, fields='id').execute()['id']
    raise SystemExit(f'「{name}」がドライブに見つかりません')
FOLDER = find('AIラジオ大学 制作室', 'application/vnd.google-apps.folder')
SHEET = find('制作ボード｜AIラジオ大学', 'application/vnd.google-apps.spreadsheet')

# 教授ごとの声の設計図（特徴を言葉で。自由に書き換えてOK）
声 = {
 '人工知能学部': '落ち着いた中年の男性の声。高さは中くらい。明快で、穏やかに語りかける。',
 '宇宙学部': 'やわらかく優しい女性の声。高さは中くらい。夜のラジオのように、静かに近い距離で語りかける。',
 '哲学部': '低めの年配の男性の声。ゆっくりで、思慮深く、穏やかに話す。',
 '心理学部': '明るく親しみやすい女性の声。高さはやや高め。抑揚が豊かで、笑顔で語りかける。',
 '神秘学部': '低く静かな男性の声。息づかいが感じられる近い距離感で、ささやくように語る。',
 '宗教学部': '穏やかで品のある年配の男性の声。落ち着いた口調で、ゆっくりと語りかける。',
 '経済学部': 'テンポのよい中年の男性の声。高さは中くらい。少しユーモラスに、楽しそうに話す。',
 '歴史学部': '温かい男性の声。高さはやや低め。語り部のように、物語を聞かせるように話す。',
}
print('準備完了')

In [ ]:
#@title ② 教授の声のオーディション（初回のみ・各5候補）
import json
from IPython.display import Audio, display, HTML
json.dump(声, open('/content/spec.json', 'w'), ensure_ascii=False)
!cd /content/Irodori-TTS && uv run --no-sync python irodori_worker.py --audition /content/spec.json /content/audition
for name in 声:
    display(HTML(f'<h3>{name}</h3>'))
    for i in range(1, 6):
        display(HTML(f'候補{i}')); display(Audio(f'/content/audition/{name}/候補{i}.wav'))

In [ ]:
#@title ③ 声を決めて保存（初回のみ・気に入った番号を選ぶ）
ai_jinkouchinou = 1  #@param [1, 2, 3, 4, 5]
uchu = 1  #@param [1, 2, 3, 4, 5]
tetsugaku = 1  #@param [1, 2, 3, 4, 5]
shinri = 1  #@param [1, 2, 3, 4, 5]
shinpi = 1  #@param [1, 2, 3, 4, 5]
shukyo = 1  #@param [1, 2, 3, 4, 5]
keizai = 1  #@param [1, 2, 3, 4, 5]
rekishi = 1  #@param [1, 2, 3, 4, 5]
from googleapiclient.http import MediaFileUpload
選んだ番号 = {'人工知能学部': ai_jinkouchinou, '宇宙学部': uchu, '哲学部': tetsugaku, '心理学部': shinri, '神秘学部': shinpi, '宗教学部': shukyo, '経済学部': keizai, '歴史学部': rekishi}
REFS = find('声の見本', 'application/vnd.google-apps.folder', FOLDER, create=True)
for name, n in 選んだ番号.items():
    old = drive.files().list(q=f"name = '{name}.wav' and '{REFS}' in parents and trashed = false", fields='files(id)').execute()['files']
    for f in old: drive.files().update(fileId=f['id'], body={'trashed': True}).execute()
    drive.files().create(body={'name': f'{name}.wav', 'parents': [REFS]}, media_body=MediaFileUpload(f'/content/audition/{name}/候補{n}.wav', mimetype='audio/wav'), fields='id').execute()
    print(f'{name}：候補{n}の声に決めました')
print('ドライブの「声の見本」フォルダに保存しました。次回からはこの声で話します。')

In [ ]:
#@title ④ 原稿を読み込む
honsu = 1  #@param {type:"integer"}
一度に作る本数 = honsu  # 1本ずつ試すのがおすすめ。慣れたら8に
import json, re, shutil, io
from pathlib import Path
from googleapiclient.http import MediaIoBaseDownload
ws = gc.open_by_key(SHEET).sheet1
rows = ws.get_all_values(); links = ws.get('E1:E200', value_render_option='FORMULA')
C = {h: i for i, h in enumerate(rows[0])}
# 決めた声（声の見本）をダウンロード
refs = {}
REFS = find('声の見本', 'application/vnd.google-apps.folder', FOLDER, create=True)
Path('/content/refs').mkdir(exist_ok=True)
for f in drive.files().list(q=f"'{REFS}' in parents and trashed = false", fields='files(id,name)').execute()['files']:
    p = f"/content/refs/{f['name']}"; fh = io.FileIO(p, 'wb'); dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=f['id']))
    done = False
    while not done: _, done = dl.next_chunk()
    fh.close(); refs[f['name'][:-4]] = p
MARK = 'ここから下が読み上げる本文'
jobs = Path('/content/jobs'); shutil.rmtree(jobs, ignore_errors=True); jobs.mkdir(); Path('/content/out').mkdir(exist_ok=True)
todo = []
for r, row in enumerate(rows[1:], start=2):
    if row[C['状態']] != '音声化待ち' or len(todo) >= 一度に作る本数: continue
    m = re.search(r'/d/([\w-]+)', str(links[r - 1][0]) if r - 1 < len(links) and links[r - 1] else '')
    if not m: print('原稿リンクが読めません:', row[C['タイトル']]); continue
    text = drive.files().export(fileId=m.group(1), mimeType='text/plain').execute().decode('utf-8-sig')
    body = text.split(MARK, 1)[1].split('\n', 1)[1] if MARK in text else text
    fac = row[C['学部']]; out = f"/content/out/{row[C['MP3ファイル名']]}"
    (jobs / f'{r:03d}.json').write_text(json.dumps({'text': body.strip(), 'caption': 声.get(fac, '落ち着いた大人の声。大学の講義のように語りかける。'), 'ref_wav': refs.get(fac), 'out_mp3': out, 'seed': 1000 + r}, ensure_ascii=False))
    todo.append((r, row[C['タイトル']], out)); print(f"音声化します：{row[C['タイトル']]}（声：{'決めた声' if fac in refs else '自動'}）")
if not todo: print('「音声化待ち」の講義はありません')

In [ ]:
#@title ⑤ 音声を作る（1本あたり数分〜）
!cd /content/Irodori-TTS && uv run --no-sync python irodori_worker.py /content/jobs

In [ ]:
#@title ⑥ ドライブに保存して記録
import datetime, os
from googleapiclient.http import MediaFileUpload
from IPython.display import Audio, display
VOICE = find('音声', 'application/vnd.google-apps.folder', FOLDER, create=True)
done = []
for row, title, out in todo:
    if not os.path.exists(out): print('できていません：', title); continue
    drive.files().create(body={'name': os.path.basename(out), 'parents': [VOICE]}, media_body=MediaFileUpload(out, mimetype='audio/mpeg'), fields='id').execute()
    ws.update_cell(row, C['状態'] + 1, '登録待ち'); ws.update_cell(row, C['Muse音声'] + 1, 'Irodori（Colab）で生成')
    done.append(title); print('保存しました：', title); display(Audio(out))
if done:
    ROOM = find('00 AI会議室｜はじめに読む', 'application/vnd.google-apps.document')
    line = f"\n{datetime.date.today()} Irodori（Colab）：{len(done)}本を音声化し、制作室の「音声」フォルダに保存しました（{'、'.join(done)}）。状態は「登録待ち」です。"
    docs.documents().batchUpdate(documentId=ROOM, body={'requests': [{'insertText': {'endOfSegmentLocation': {}, 'text': line}}]}).execute()
    print('ボードと会議ログを更新しました。スマホのアプリで「研究室 → MP3を追加」から登録してください。')